# 6. Deployment: Implementasi dan Diseminasi Sistem Geospasial Tutupan Lahan Sentinel-2 Jawa Timur

**Mata Kuliah**: Proyek Sains Data  
**Kerangka Kerja**: CRISP-DM (*Cross-Industry Standard Process for Data Mining*)  
**Wilayah Studi**: Provinsi Jawa Timur, Indonesia  
**Sensor Citra**: Satelit Multi-Spektral Copernicus Sentinel-2 Level-2A  
**Total Data Sampel**: 250 Sampel Poligon Lapangan (50 Sampel per Kelas Seimbang)  
**Antarmuka Web**: Streamlit Interactive Dashboard (`app.py`)  

---

## 6.1 Tujuan dan Peran Tahap Deployment dalam CRISP-DM

Tahap **Deployment** merupakan fase pamungkas dalam metodologi CRISP-DM. Pada tahap ini, seluruh pengetahuan data (*data understanding*), rekayasa fitur biofisik (*data preparation*), model pembelajaran mesin terbaik (*modeling*), dan metrik validasi ilmiah (*evaluation*) ditransformasikan dari bentuk eksperimen statis (*notebook*) menjadi **produk perangkat lunak interaktif yang siap pakai (*production-ready interactive software*)**.

Tujuan utama tahap deployment pada proyek ini mencakup:
1. **Aksesibilitas & Diseminasi Praktis**: Memungkinkan pembuat kebijakan lingkungan, analis GIS, dan akademisi mengakses hasil prediksi klasifikasi tutupan lahan (*Land Use / Land Cover* / LULC) tanpa perlu menjalankan baris kode pemrograman secara manual.
2. **Visualisasi Spasial Tanpa Desktop Proprietary GIS**: Seluruh visualisasi spasial, perlapisan layer poligon ground-truth, dan citra satelit resolusi tinggi disajikan secara interaktif menggunakan pustaka *open-source* Python (**Folium / Leaflet** dan **Streamlit**), mematuhi batasan proyek **tanpa ketergantungan pada software desktop QGIS**.
3. **Penyediaan Pipeline Inferensi Terpadu (*End-to-End Inference*)**: Mengintegrasikan alur **Ambil $\rightarrow$ Proses $\rightarrow$ Klasifikasi $\rightarrow$ Dapet Informasi** dalam satu sistem terpadu yang mampu memproses koordinat geospasial riil, input slider manual band satelit, maupun unggahan foto citra lapangan.


## 6.2 Arsitektur Sistem dan Komponen Teknologi

Arsitektur aplikasi deployment dirancang dengan prinsip modularitas, skalabilitas, dan efisiensi komputasi:

```
[ Pengguna / Browser ]
        │
        ▼ (Port 8501)
┌────────────────────────────────────────────────────────────────────────┐
│                   Aplikasi Web Streamlit (app.py)                      │
│                                                                        │
│  ┌───────────────────────┐  ┌───────────────────────────────────────┐  │
│  │ Tab 1: Peta Leaflet   │  │ Tab 2: Live Inference Engine          │  │
│  │ - Basemap Satelit     │  │ - 3 Mode Input (Upload/Preset/Slider) │  │
│  │ - Layer Filter        │  │ - OpenCV Land Feature Segmentation    │  │
│  │ - Sample Navigator    │  │ - 22 Biophysical Indices Pipeline     │  │
│  │ - Circle Markers      │  │ - Model Ensemble Soft Voting          │  │
│  │ - Direktori Wilayah   │  │ - Physical Spectral Interpretation    │  │
│  └───────────────────────┘  └───────────────────────────────────────┘  │
│  ┌───────────────────────┐  ┌───────────────────────────────────────┐  │
│  │ Tab 3: Profil Spektral│  │ Tab 4: Metrik & Evaluasi CRISP-DM     │  │
│  │ - 7-Band Signature   │  │ - KPI Summary (86.0% OA, 0.825 Kappa) │  │
│  │ - 22 Index Matrix     │  │ - Producer / User / OvR Accuracy Table│  │
│  └───────────────────────┘  └───────────────────────────────────────┘  │
└────────────────────────────────────┬───────────────────────────────────┘
                                     │ Memuat Dependensi & Data
                                     ▼
┌────────────────────────────────────────────────────────────────────────┐
│                        Data & Model Artifacts                          │
│                                                                        │
│ • Model: models/model_terbaik.pkl (Weighted Soft Voting Ensemble)      │
│ • Metadata: models/model_metadata.json (29 Feature Schema)             │
│ • GeoJSON: sampling_geojson/ (5 File Poligon Ground-Truth Jawa Timur)  │
│ • Dataset: prepared_data/data_prepared_all.csv (250 Sampel / 29 Fitur) │
│ • Normalisasi: prepared_data/standard_scaler.pkl                       │
└────────────────────────────────────────────────────────────────────────┘
```

### Rincian Pustaka Perangkat Lunak:
* **Streamlit (`streamlit`)**: Kerangka kerja pengembangan web berbasis Python untuk membuat antarmuka pengguna analitik secara reaktif.
* **Folium & Streamlit-Folium (`folium`, `streamlit-folium`)**: Rendering peta interaktif berbasis Leaflet.js dengan kontrol layer dinamis dan penanda kustom.
* **OpenCV (`cv2`)**: Analisis citra komputasional (*computer vision*) untuk segmentasi daratan, pembersihan langit (*sky masking*), dan ekstraksi tepi akar mangrove/struktur bangunan.
* **Plotly (`plotly.express`, `plotly.graph_objects`)**: Grafik interaktif kurva pantulan spektral 7 band dan diagram probabilitas kelas.
* **Scikit-Learn & Joblib (`sklearn`, `joblib`)**: Pemuatan dan eksekusi model *ensemble* pembelajaran mesin.


## 6.3 Model Pembelajaran Mesin yang Dideploy

Model yang diintegrasikan ke dalam antarmuka aplikasi deployment adalah **Weighted Soft Voting Ensemble** (`model_terbaik.pkl`), yang merupakan model terbaik dari pengujian pada tahap Modeling dan Evaluation.

### Karakteristik Model:
1. **Algoritma Anggota Ansambel**:
   * **Random Forest Classifier**: Mengurangi variansi galat dan stabil terhadap pencilan spektral melalui agregasi 100 pohon keputusan acak.
   * **LightGBM Classifier**: *Gradient boosting* berbasis histogram daun (*leaf-wise tree growth*) yang sangat cepat dan sensitif pada batas pemisah kelas.
   * **XGBoost Classifier**: *Extreme Gradient Boosting* dengan regularisasi L1/L2 untuk mencegah *overfitting* pada fitur-fitur berkolinearitas tinggi.
   * **Extra Trees Classifier**: Mengacak pemisahan ambang batas (*threshold split*) untuk menangani variansi heterogen pada vegetasi campuran.
2. **Skema Keputusan Voting Lunak (*Soft Voting*)**:
   Model menghitung rata-rata tertimbang dari distribusi probabilitas posterior yang diprediksi oleh masing-masing estimator:
   $$\hat{P}(y = c \mid \mathbf{x}) = \sum_{m=1}^{M} w_m P_m(y = c \mid \mathbf{x})$$
   di mana $w_m$ adalah bobot kontribusi model ke-$m$ dan $c \in \{0, 1, 2, 3, 4\}$.
3. **Dimensi Fitur Masukan (29 Fitur Terpadu)**:
   * **7 Band Spektral Asli Sentinel-2 L2A**: `B02` (Blue), `B03` (Green), `B04` (Red), `B08` (NIR Broad), `B8A` (NIR Narrow), `B11` (SWIR-1), `B12` (SWIR-2).
   * **22 Indeks Biofisik Spektral**: `NDVI, MNDWI, NDBI, SAVI, MSAVI, BSI, UI, EBBI, NDBI_NDVI_Diff, EVI, GNDVI, NDMI_Narrow, SWIR_Ratio, Soil_Iron_Oxide, AWEI_nsh, TC_Wetness, Mangrove_Substrate_Moisture, IBI, VR, Urban_Impervious_Ratio, B8A_B11_Ratio, Mangrove_Index_8A_11_4`.
4. **Kinerja yang Terbukti pada Tahap Evaluasi**:
   * **Overall Accuracy (OA)**: **86.00%** pada 50 data uji independen.
   * **Macro ROC-AUC**: **97.15%**.
   * **Koefisien Cohen's Kappa**: **0.825** (*Substantial Agreement*).
   * **Akurasi Per Kelas (Recall)**: Mangrove **100.00%**, Hutan Daratan **100.00%**, Perairan **90.00%**, Pertanian **70.00%**, Bangunan **70.00%**.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import joblib
import numpy as np
import pandas as pd

# Konfigurasi path relatif terhadap root Tugas_UTS
BASE_DIR = os.path.abspath("")
UTS_DIR = os.path.dirname(BASE_DIR) if os.path.basename(BASE_DIR) == "6-deployment" else os.path.join(BASE_DIR, "src", "Tugas_UTS")
MODEL_DIR = os.path.join(UTS_DIR, "4-modeling", "models")
DATA_DIR = os.path.join(UTS_DIR, "3-data_preparation", "prepared_data")

model_path = os.path.join(MODEL_DIR, "model_terbaik.pkl")
meta_path = os.path.join(MODEL_DIR, "model_metadata.json")

# Muat Model dan Metadata Fitur
model = joblib.load(model_path)
with open(meta_path, "r") as f:
    metadata = json.load(f)

feature_cols = metadata["feature_cols"]

print("=" * 70)
print(f"STATUS ARTIFAK MODEL DEPLOYMENT:")
print(f"• Tipe Model             : {type(model).__name__}")
print(f"• File Model             : {os.path.basename(model_path)} ({os.path.getsize(model_path)/1024:.1f} KB)")
print(f"• Jumlah Fitur Terpadu   : {len(feature_cols)} Fitur")
print(f"• Daftar 7 Band Masukan  : {feature_cols[:7]}")
print(f"• Daftar 22 Indeks       : {feature_cols[7:]}")
print("=" * 70)


STATUS ARTIFAK MODEL DEPLOYMENT:
• Tipe Model             : VotingClassifier
• File Model             : model_terbaik.pkl (14012.9 KB)
• Jumlah Fitur Terpadu   : 29 Fitur
• Daftar 7 Band Masukan  : ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
• Daftar 22 Indeks       : ['NDVI', 'MNDWI', 'NDBI', 'B8A_B11_Ratio', 'Mangrove_Index_8A_11_4', 'SAVI', 'MSAVI', 'BSI', 'UI', 'EBBI', 'NDBI_NDVI_Diff', 'EVI', 'GNDVI', 'NDMI_Narrow', 'SWIR_Ratio', 'Soil_Iron_Oxide', 'AWEI_nsh', 'TC_Wetness', 'Mangrove_Substrate_Moisture', 'IBI', 'VR', 'Urban_Impervious_Ratio']


## 6.4 Pipeline Rekayasa Fitur Otomatis (*Feature Engineering Engine*)

Ketika pengguna memasukkan 7 band reflektansi satelit (baik melalui unggahan citra, pemilihan sampel, maupun slider), sistem secara otomatis mengeksekusi pipeline matematika biofisik berikut untuk menghasilkan 22 indeks spektral secara *real-time*:

$$\text{NDVI} = \frac{B08 - B04}{B08 + B04}, \quad \text{MNDWI} = \frac{B03 - B11}{B03 + B11}, \quad \text{NDBI} = \frac{B11 - B08}{B11 + B08}$$
$$\text{Mangrove Index} = \frac{B8A - B04}{B11}, \quad \text{EVI} = 2.5 \times \frac{B08 - B04}{B08 + 6 B04 - 7.5 B02 + 1}$$
$$\text{EBBI} = \frac{B11 - B08}{10 \sqrt{B11 + B08}}, \quad \text{TC Wetness} = 0.151 B02 + 0.197 B03 + 0.328 B04 + 0.341 B08 - 0.712 B11 - 0.456 B12$$


In [2]:
def compute_spectral_indices(b02, b03, b04, b08, b8a, b11, b12):
    eps = 1e-6
    # 1. Indeks Dasar Spektral
    ndvi = (b08 - b04) / (b08 + b04 + eps)
    mndwi = (b03 - b11) / (b03 + b11 + eps)
    ndbi = (b11 - b08) / (b11 + b08 + eps)
    savi = ((b08 - b04) * 1.5) / (b08 + b04 + 0.5 + eps)
    msavi = (2 * b08 + 1 - np.sqrt(np.maximum((2 * b08 + 1)**2 - 8 * (b08 - b04), 0))) / 2.0
    mangrove_idx = (b8a - b04) / (b11 + eps)
    bsi = ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + eps)
    ui = (b12 - b08) / (b12 + b08 + eps)
    ebbi = (b11 - b08) / (10.0 * np.sqrt(np.maximum(b11 + b08, 0.0) + eps))
    ndbi_ndvi_diff = ndbi - ndvi

    # 2. Indeks Lanjutan Biofisik Spektral
    b8a_b11_ratio = b8a / (b11 + eps)
    evi = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + eps)
    gndvi = (b08 - b03) / (b08 + b03 + eps)
    ndmi_narrow = (b8a - b11) / (b8a + b11 + eps)
    swir_ratio = b11 / (b12 + eps)
    soil_iron_oxide = b04 / (b02 + eps)
    awei_nsh = 4.0 * (b03 - b11) - (0.25 * b08 + 2.75 * b12)
    tc_wetness = 0.1511 * b02 + 0.1973 * b03 + 0.3283 * b04 + 0.3407 * b08 - 0.7117 * b11 - 0.4559 * b12
    mangrove_substrate_moisture = (b8a - b12) / (b8a + b12 + eps)
    ibi = (ndbi - (savi + mndwi)/2.0) / (ndbi + (savi + mndwi)/2.0 + eps)
    vr = b08 / (b04 + eps)
    urban_impervious_ratio = b11 / (b02 + eps)

    return {
        'B02': b02, 'B03': b03, 'B04': b04, 'B08': b08, 'B8A': b8a, 'B11': b11, 'B12': b12,
        'NDVI': ndvi, 'MNDWI': mndwi, 'NDBI': ndbi, 'B8A_B11_Ratio': b8a_b11_ratio,
        'Mangrove_Index_8A_11_4': mangrove_idx, 'SAVI': savi, 'MSAVI': msavi,
        'BSI': bsi, 'UI': ui, 'EBBI': ebbi, 'NDBI_NDVI_Diff': ndbi_ndvi_diff,
        'EVI': evi, 'GNDVI': gndvi, 'NDMI_Narrow': ndmi_narrow, 'SWIR_Ratio': swir_ratio,
        'Soil_Iron_Oxide': soil_iron_oxide, 'AWEI_nsh': awei_nsh,
        'TC_Wetness': tc_wetness, 'Mangrove_Substrate_Moisture': mangrove_substrate_moisture,
        'IBI': ibi, 'VR': vr, 'Urban_Impervious_Ratio': urban_impervious_ratio
    }

# Uji Coba Fungsi pada Sampel Reflektansi Hutan Mangrove
sample_mng = compute_spectral_indices(0.100, 0.117, 0.091, 0.314, 0.330, 0.147, 0.087)
print("HASIL KOMPUTASI 22 INDEKS PADA SAMPEL MANGROVE:")
print(f"• NDVI                     : {sample_mng['NDVI']:.4f} (Vegetasi Tajuk Lebat)")
print(f"• MNDWI                    : {sample_mng['MNDWI']:.4f} (Kadar Air Permukaan)")
print(f"• Mangrove Index (8A-11-4) : {sample_mng['Mangrove_Index_8A_11_4']:.4f} (Tinggi / Khas Mangrove)")
print(f"• SWIR Ratio (B11/B12)     : {sample_mng['SWIR_Ratio']:.4f}")
print(f"• TC Wetness               : {sample_mng['TC_Wetness']:.4f}")


HASIL KOMPUTASI 22 INDEKS PADA SAMPEL MANGROVE:
• NDVI                     : 0.5506 (Vegetasi Tajuk Lebat)
• MNDWI                    : -0.1136 (Kadar Air Permukaan)
• Mangrove Index (8A-11-4) : 1.6258 (Tinggi / Khas Mangrove)
• SWIR Ratio (B11/B12)     : 1.6896
• TC Wetness               : 0.0308


## 6.5 Simulasi Inferensi End-to-End pada 5 Kelas Tutupan Lahan

Berikut adalah simulasi eksekusi inferensi model pada kelima kelas tutupan lahan Jawa Timur untuk memvalidasi bahwa model memprediksi masing-masing kelas dengan tingkat keyakinan (*confidence score*) yang tinggi:


In [3]:
CLASS_NAMES = {
    0: "Bangunan (Built-up)",
    1: "Lahan Pertanian (Farmland)",
    2: "Perairan (Water Body)",
    3: "Hutan Daratan (Terrestrial Forest)",
    4: "Hutan Mangrove (Mangrove Forest)"
}

test_scenarios = {
    "Skenario 1 - Kawasan Perkotaan / Bangunan": (0.168, 0.184, 0.207, 0.249, 0.268, 0.309, 0.291),
    "Skenario 2 - Hamparan Sawah Irigasi": (0.077, 0.104, 0.113, 0.244, 0.259, 0.219, 0.166),
    "Skenario 3 - Danau / Laut Terbuka": (0.106, 0.114, 0.098, 0.086, 0.076, 0.071, 0.064),
    "Skenario 4 - Kanopi Hutan Pegunungan": (0.165, 0.176, 0.167, 0.336, 0.354, 0.231, 0.187),
    "Skenario 5 - Zona Pasang-Surut Bakau": (0.100, 0.117, 0.091, 0.314, 0.330, 0.147, 0.087)
}

results = []
for label, bands in test_scenarios.items():
    feats = compute_spectral_indices(*bands)
    input_df = pd.DataFrame([feats])[feature_cols]
    
    pred_class = int(model.predict(input_df)[0])
    prob_dist = model.predict_proba(input_df)[0]
    conf_score = prob_dist[pred_class] * 100
    
    results.append({
        "Skenario Pengujian": label,
        "Kelas Terprediksi": CLASS_NAMES[pred_class],
        "Tingkat Keyakinan (%)": f"{conf_score:.2f}%",
        "NDVI": f"{feats['NDVI']:.3f}",
        "MNDWI": f"{feats['MNDWI']:.3f}",
        "NDBI": f"{feats['NDBI']:.3f}",
        "Mangrove Idx": f"{feats['Mangrove_Index_8A_11_4']:.3f}"
    })

sim_df = pd.DataFrame(results)
display(sim_df)


,Skenario Pengujian,Kelas Terprediksi,Tingkat Keyakinan (%),NDVI,MNDWI,NDBI,Mangrove Idx
0,Skenario 1 - Kawasan Perkotaan / Bangunan,Bangunan (Built-up),99.60%,0.092,-0.254,0.108,0.197
1,Skenario 2 - Hamparan Sawah Irigasi,Lahan Pertanian (Farmland),93.26%,0.367,-0.356,-0.054,0.667
2,Skenario 3 - Danau / Laut Terbuka,Perairan (Water Body),99.49%,-0.065,0.232,-0.096,-0.310
3,Skenario 4 - Kanopi Hutan Pegunungan,Hutan Daratan (Terrestrial Forest),75.61%,0.336,-0.135,-0.185,0.810
4,Skenario 5 - Zona Pasang-Surut Bakau,Hutan Mangrove (Mangrove Forest),86.08%,0.551,-0.114,-0.362,1.626


## 6.6 Rincian Fitur Utama Aplikasi Web Deployment (`app.py`)

Aplikasi web yang dibangun pada file `src/Tugas_UTS/6-deployment/app.py` memiliki 4 tab utama dengan fungsionalitas komprehensif:

### Tab 1: Peta Interaktif Citra Satelit (Leaflet & OpenStreetMap / Esri)
1. **Perlapisan 5 Layer Tematik (*Thematic Layer Filtering*)**:
   Checkbox pada sidebar untuk mengaktifkan/menonaktifkan masing-masing dari 5 kelas tutupan lahan secara terpisah. Fitur ini secara khusus memenuhi instruksi ujian untuk memfilter lahan pertanian/sawah secara mandiri.
2. **Navigator Pencarian & Fokus Poligon Sampel (*Auto-Centering & Zoom 16*)**:
   Menyelesaikan tantangan kesulitan menemukan poligon kecil (bangunan, petak sawah, dan mangrove) di peta luas Provinsi Jawa Timur. Pengguna dapat memilih ID sampel tertentu (misalnya `mangrove_001` atau `bangunan_004`), dan kamera satelit akan **otomatis melompat dan memperbesar hingga tingkat zoom 16 (sub-100 meter)** disertai pin marker merah dan cincin sorotan kuning keemasan.
3. **Titik Penanda Melingkar (*Circle Markers*) Seluruh 250 Sampel**:
   Menampilkan lingkaran berwarna jelas pada seluruh sampel ground-truth sehingga sebaran klaster di Jawa Timur (klaster mangrove Gresik/Surabaya, sawah Lamongan/Bojonegoro, bangunan Surabaya/Malang) terlihat jelas dari skala provinsi.
4. **Legenda Tematik LULC & Direktori Wilayah Geografis**:
   Dilengkapi tabel pencarian interaktif koordinat Lintang/Bujur dan nilai spektral 250 sampel lapangan.

---

### Tab 2: Alur Inferensi Prediksi Multimoda (Live Prediction Engine)
Menerapkan paradigma **Ambil $\rightarrow$ Proses $\rightarrow$ Klasifikasi $\rightarrow$ Dapet Informasi**:
1. **Mode 1: Unggah Foto Citra Lapangan (`st.file_uploader`)**:
   Dilengkapi modul **Computer Vision OpenCV** dan **Sky/Cloud Masking** cerdas. Sistem memfilter area langit berkabut/awan terang di bagian atas foto, menganalisis indeks kehijauan (*Excess Green Index*), tekstur akar tunjang mangrove atau tepi geometris bangunan, dan memetakan ke profil spektral fisik Sentinel-2 BOA yang tepat.
2. **Mode 2: Preset Sampel Lapangan Jawa Timur**:
   Memungkinkan penguji memilih secara instan dari 250 data riil lapangan untuk memvalidasi ground-truth kelas.
3. **Mode 3: Slider Manual 7 Band Sentinel-2**:
   Memungkinkan peneliti melakukan eksperimen skenario reflektansi fisik BOA.
4. **Output Prediksi & Interpretasi Fisika Spektral**:
   Menampilkan nama kelas terprediksi, kartu probabilitas visual horizontal, dan ulasan biofisik (misal mengapa mangrove terpisah dari hutan daratan berdasarkan penyerapan SWIR pada substrat lumpur).

---

### Tab 3: Analisis Profil Spektral dan Matriks 22 Indeks Biofisik
* **Kurva Pantulan Spektral Rata-rata 7 Band Sentinel-2**: Grafik interaktif Plotly yang memperlihatkan perbedaan pola pantulan (*reflectance signature*) masing-masing kelas dari spektrum tampak hingga inframerah gelombang pendek.
* **Matriks Komparasi 22 Indeks Biofisik Terpadu**: Tabel rata-rata nilai indeks biofisik (NDVI, EVI, GNDVI, MNDWI, NDBI, SAVI, BSI, UI, EBBI, IBI, dll.) per kelas tutupan lahan.

---

### Tab 4: Dasbor Metrik Evaluasi dan Kinerja Ilmiah CRISP-DM
* **Ringkasan KPI Model**: Total 250 sampel seimbang, 29 fitur terpadu, Overall Accuracy 86.00%, dan Macro ROC-AUC 97.15%.
* **Tabel Rinci Akurasi Per Kelas**:
  * **Akurasi Per Kelas / Producer's Accuracy (Recall)**: Mengukur akurasi dari sudut pandang pembuat peta (seberapa baik area di lapangan terpetakan).
  * **Akurasi Pengguna / User's Accuracy (Precision)**: Mengukur keandalan peta bagi pengguna akhir.
  * **Akurasi Biner One-vs-Rest (OvR)**: Mengukur ketepatan dikotomi biner kelas terhadap kelas lainnya.
  * **F1-Score**: Rata-rata harmonik antara presisi dan recall.
* **Kesimpulan Ilmiah Pemetaan**: Sintesis rekomendasi pemilihan fitur dan mitigasi kebingungan spektral.


In [4]:
# Verifikasi Struktur Berkas dan Integritas Sistem Deployment
app_path = os.path.join(UTS_DIR, "6-deployment", "app.py")
geojson_dir = os.path.join(UTS_DIR, "2-data_understanding", "sampling_geojson")

print("=" * 70)
print("VERIFIKASI INTEGRITAS FILE DEPLOYMENT:")
print(f"• File Utama Aplikasi Web : {os.path.basename(app_path)} -> {'ADA' if os.path.exists(app_path) else 'TIDAK ADA'}")
print(f"• Ukuran Kode app.py      : {os.path.getsize(app_path)/1024:.1f} KB")

geojson_files = [f for f in os.listdir(geojson_dir) if f.endswith(('.geojson', '.json'))]
print(f"• File Poligon GeoJSON    : {len(geojson_files)} file terverifikasi ({geojson_files})")

data_all_path = os.path.join(DATA_DIR, "data_prepared_all.csv")
print(f"• Dataset 250 Sampel      : {'ADA' if os.path.exists(data_all_path) else 'TIDAK ADA'} ({os.path.getsize(data_all_path)/1024:.1f} KB)")
print("=" * 70)
print("Seluruh komponen deployment terverifikasi lengkap dan siap dijalankan.")


VERIFIKASI INTEGRITAS FILE DEPLOYMENT:
• File Utama Aplikasi Web : app.py -> ADA
• Ukuran Kode app.py      : 56.8 KB
• File Poligon GeoJSON    : 5 file terverifikasi (['bangunan.geojson', 'hutan.geojson', 'mangrove.json', 'perairan.geojson', 'pertanian.geojson'])
• Dataset 250 Sampel      : ADA (139.4 KB)
Seluruh komponen deployment terverifikasi lengkap dan siap dijalankan.


## 6.7 Panduan Menjalankan Aplikasi Web

Aplikasi web dapat dijalankan secara lokal maupun diakses langsung secara daring (*online*):

### 1. Menjalankan Secara Lokal (*Local Machine*):
1. Buka terminal atau PowerShell pada direktori root proyek.
2. Jalankan perintah eksekusi Streamlit:
   ```bash
   streamlit run src/Tugas_UTS/6-deployment/app.py
   ```
3. Akses antarmuka aplikasi melalui peramban web (*web browser*) pada alamat:
   * **Local URL**: <a href="http://localhost:8501" target="_blank" rel="noopener noreferrer">http://localhost:8501 ↗</a>

### 2. Akses Aplikasi Live Deployment Online (*Streamlit Cloud*):
Aplikasi web klasifikasi tutupan lahan ini telah berhasil dideploy secara publik di cloud dan dapat langsung diakses melalui tautan berikut:

* **Live Deployment URL**: <div style="margin-top: 12px; margin-bottom: 12px;">
  <a href="https://sentinel2-jatim-lulc-app-53k6uykemnmygpqkcbyafm.streamlit.app/" target="_blank" rel="noopener noreferrer" style="display: inline-block; padding: 10px 20px; background-color: #ff4b4b; color: white; font-weight: bold; border-radius: 8px; text-decoration: none; box-shadow: 0 2px 5px rgba(0,0,0,0.15);">
    Buka Aplikasi Live Streamlit Deployment
  </a>
</div>


<!-- <a href="https://sentinel2-jatim-lulc-app-53k6uykemnmygpqkcbyafm.streamlit.app/" target="_blank" rel="noopener noreferrer">**https://sentinel2-jatim-lulc-app-53k6uykemnmygpqkcbyafm.streamlit.app/** ↗ (Klik untuk membuka di tab baru)</a> -->



---

## 6.8 Kesimpulan Tahap Deployment

Implementasi tahap Deployment ini berhasil menyelesaikan seluruh siklus 6 tahap metodologi **CRISP-DM** (*Business Understanding $\rightarrow$ Data Understanding $\rightarrow$ Data Preparation $\rightarrow$ Modeling $\rightarrow$ Evaluation $\rightarrow$ Deployment*). 

Sistem geospasial yang dihasilkan tidak hanya unggul secara akademis dan teoritis dengan akurasi pengujian **86.00%**, melainkan juga siap didiseminasikan dalam bentuk instrumen pemantauan lingkungan hidup yang interaktif, transparan, dan mudah dioperasikan.